## importing the required libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.transforms import ToTensor
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader


In [2]:
# data preparation
# transform convert images to pytorch tensors and normalize pixels to[0,1]

transform = transforms.Compose([transforms.ToTensor(),])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

In [3]:
train_dataset

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
           )

In [4]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten() # flatten the 28*28 images into 784 pixels
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 128),   # hidden layer 1
            nn.ReLU(),              # activation function
            nn.Linear(128, 10),     # output layer (10 classes for digits 0-9)
        )
        
    def forward(self, x):
        x=self.flatten(x)
        logits=self.linear_relu_stack(x)
        return logits
            

In [6]:
# initialize model,loss,and optimizer
device = "cuda" if torch.cuda.is_available() else "cpu"
model = NeuralNetwork().to(device)
criterion = nn.CrossEntropyLoss()   # standard for multiclass classification
optimizer = optim.Adam(model.parameters(),lr = 0.001)

In [7]:
#  training loop
epochs = 5
print(f"training pytorch model  on {device}:")

for epoch in range(epochs):
    model.train()
    running_loss =0.0
    for X,y in train_loader:
        X,y = X.to(device),y.to(device)
        
        # forward pass 
        
        pred = model(X)
        loss = criterion(pred,y)
        
        # backward propagation 
        optimizer.zero_grad()   # clear gradient from previous step
        loss.backward()     # compute new gradient
        optimizer.step()  # update network weights
        
        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f}")
        

training pytorch model  on cpu:
Epoch 1/5 | Loss: 0.3418
Epoch 2/5 | Loss: 0.1568
Epoch 3/5 | Loss: 0.1086
Epoch 4/5 | Loss: 0.0818
Epoch 5/5 | Loss: 0.0649
